# DistilBERT (Colab) — Experiment D (Preprocessing + Imbalance Sweep)

This notebook benchmarks:

- **Preprocessing strategy**
  - `baseline` (no train rebalancing)
  - `rebalanced` (train-only downsample/oversample)
- **Imbalance loss setting**
  - `BCEWithLogitsLoss` (no `pos_weight`)
  - `BCEWithLogitsLoss(pos_weight)` clipped to `[1.0, 20.0]`
  - `BCEWithLogitsLoss(pos_weight)` clipped to `[1.0, 50.0]`

Model and core setup:

- `distilbert-base-uncased`
- `problem_type="multi_label_classification"`
- labels: `LABEL_COLUMNS`
- iterative multilabel stratified split (`validation_fraction=0.1`, `random_state=42`)
- benchmark includes baseline preprocessing and train-only rebalanced preprocessing
- exports performance, efficiency, and hyperparameter metadata so best settings are traceable.

In [ ]:
# Colab setup: install dependencies
!pip -q install torch transformers pandas numpy matplotlib scikit-learn seaborn iterative-stratification

In [ ]:
# Mount Google Drive and set paths
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'artifacts_colab_exp_d'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

In [ ]:
import contextlib
import math
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels, threshold_grid: np.ndarray):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 3), 'best_f1_on_val': best_f1})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 50.0) -> torch.Tensor:
    y = np.asarray(y_train, dtype=np.float32)
    positives = y.sum(axis=0)
    total = float(y.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


def make_confusion_artifacts(y_true: np.ndarray, y_pred: np.ndarray, labels):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)

    per_label_rows = []
    for j, label in enumerate(labels):
        cm = confusion_matrix(y_true[:, j], y_pred[:, j], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        per_label_rows.append({'label': label, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})

    per_label_df = pd.DataFrame(per_label_rows)

    agg_cm = confusion_matrix(y_true.ravel(), y_pred.ravel(), labels=[0, 1])
    agg_tn, agg_fp, agg_fn, agg_tp = agg_cm.ravel()
    aggregate_df = pd.DataFrame([
        {'tn': int(agg_tn), 'fp': int(agg_fp), 'fn': int(agg_fn), 'tp': int(agg_tp)}
    ])

    return per_label_df, aggregate_df, agg_cm


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


def enc_dict(enc):
    keys = [k for k in ('input_ids', 'attention_mask') if k in enc]
    return {k: enc[k] for k in keys}

In [ ]:
# Experiment D configuration (requested settings)
DEVICE = pick_device()
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

torch.manual_seed(42)
np.random.seed(42)

MODEL_NAME = 'distilbert-base-uncased'
PROBLEM_TYPE = 'multi_label_classification'
VALIDATION_FRACTION = 0.1
RANDOM_STATE = 42

MAX_LENGTH = 128
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
MAX_EPOCHS = 5
LR = 1.75e-5
WEIGHT_DECAY = 0.015
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 0.0

USE_BF16 = True
USE_TORCH_COMPILE = False
NUM_WORKERS = 2
SKIP_COMPLETED = True

THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None

# Preprocessing benchmark variants
PREPROCESSING_VARIANTS = [
    {
        'preprocessing_variant': 'baseline',
        'rebalance_train': False,
        'clean_to_toxic_ratio': 3.0,
        'rare_oversample_factor': 1.0,
        'max_copies_per_row': 3,
    },
    {
        'preprocessing_variant': 'rebalanced',
        'rebalance_train': True,
        'clean_to_toxic_ratio': 3.0,
        'rare_oversample_factor': 1.5,
        'max_copies_per_row': 3,
    },
]

# Imbalance loss sweep variants
LOSS_VARIANTS = [
    {'loss_variant': 'bce_no_pos_weight', 'use_pos_weight': False, 'pos_weight_clip_max': None},
    {'loss_variant': 'bce_pos_weight_clip20', 'use_pos_weight': True, 'pos_weight_clip_max': 20.0},
    {'loss_variant': 'bce_pos_weight_clip50', 'use_pos_weight': True, 'pos_weight_clip_max': 50.0},
]

_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('USE_BF16 requested but bf16 not supported on this GPU; training in full precision.')

print('Device:', DEVICE)
print('AMP (bf16):', USE_AMP)
print('TF32 enabled on CUDA:', DEVICE.type == 'cuda')
print('Total benchmark runs:', len(PREPROCESSING_VARIANTS) * len(LOSS_VARIANTS))

In [ ]:
# Run Experiment D sweep: preprocessing x imbalance-loss variants
summary_rows = []
per_label_frames = []
threshold_frames = []
conf_per_label_base_frames = []
conf_per_label_tuned_frames = []
conf_agg_base_rows = []
conf_agg_tuned_rows = []

pin = DEVICE.type == 'cuda'

run_specs = []
for p in PREPROCESSING_VARIANTS:
    for l in LOSS_VARIANTS:
        run_specs.append({**p, **l})

for idx, spec in enumerate(run_specs, start=1):
    run_id = f"{spec['preprocessing_variant']}__{spec['loss_variant']}"
    row_file = ARTIFACT_DIR / f'summary_{run_id}.csv'
    per_label_file = ARTIFACT_DIR / f'per_label_{run_id}.csv'
    thresholds_file = ARTIFACT_DIR / f'thresholds_{run_id}.csv'
    conf_per_label_base_file = ARTIFACT_DIR / f'confusion_per_label_baseline_{run_id}.csv'
    conf_per_label_tuned_file = ARTIFACT_DIR / f'confusion_per_label_tuned_{run_id}.csv'
    conf_agg_base_file = ARTIFACT_DIR / f'confusion_aggregate_baseline_{run_id}.csv'
    conf_agg_tuned_file = ARTIFACT_DIR / f'confusion_aggregate_tuned_{run_id}.csv'

    files_exist = all([
        row_file.exists(), per_label_file.exists(), thresholds_file.exists(),
        conf_per_label_base_file.exists(), conf_per_label_tuned_file.exists(),
        conf_agg_base_file.exists(), conf_agg_tuned_file.exists()
    ])

    if SKIP_COMPLETED and files_exist:
        print(f"[{idx}/{len(run_specs)}] {run_id}: skipping (already exists)")
        summary_rows.append(pd.read_csv(row_file).iloc[0].to_dict())
        per_label_frames.append(pd.read_csv(per_label_file))
        threshold_frames.append(pd.read_csv(thresholds_file))
        conf_per_label_base_frames.append(pd.read_csv(conf_per_label_base_file))
        conf_per_label_tuned_frames.append(pd.read_csv(conf_per_label_tuned_file))
        conf_agg_base_rows.append(pd.read_csv(conf_agg_base_file).iloc[0].to_dict())
        conf_agg_tuned_rows.append(pd.read_csv(conf_agg_tuned_file).iloc[0].to_dict())
        continue

    print(f"[{idx}/{len(run_specs)}] {run_id}: running")

    run_data = preprocess_for_distilbert(
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        max_train_samples=MAX_TRAIN_SAMPLES,
        max_val_samples=MAX_VAL_SAMPLES,
        use_iterative_stratify=True,
        rebalance_train=spec['rebalance_train'],
        clean_to_toxic_ratio=spec['clean_to_toxic_ratio'],
        rare_labels=('severe_toxic', 'threat', 'identity_hate'),
        rare_oversample_factor=spec['rare_oversample_factor'],
        max_copies_per_row=spec['max_copies_per_row'],
        rebalance_random_state=RANDOM_STATE,
        print_diagnostics=True,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc = enc_dict(run_data.val_encodings)
    y_train = np.asarray(run_data.y_train, dtype=np.float32)
    y_val = np.asarray(run_data.y_val, dtype=np.int64)

    train_loader = DataLoader(
        EncDataset(train_enc, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        EncDataset(val_enc, y_val),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLUMNS),
        problem_type=PROBLEM_TYPE,
    ).to(DEVICE)

    if USE_TORCH_COMPILE and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)  # type: ignore[assignment]
            print('  torch.compile enabled')
        except Exception as e:
            print('  torch.compile skipped:', e)

    num_params = torch_parameter_count(model)

    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

    if spec['use_pos_weight']:
        pos_weight = compute_pos_weight(y_train, max_weight=float(spec['pos_weight_clip_max'])).to(DEVICE)
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        pos_weight_mean = float(pos_weight.detach().cpu().mean().item())
        pos_weight_max = float(pos_weight.detach().cpu().max().item())
    else:
        pos_weight = None
        loss_fn = torch.nn.BCEWithLogitsLoss()
        pos_weight_mean = float('nan')
        pos_weight_max = float('nan')

    steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    num_training_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(WARMUP_RATIO * num_training_steps)
    warmup_steps = max(0, min(warmup_steps, num_training_steps - 1)) if num_training_steps > 0 else 0
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )

    train_time_s = 0.0
    inference_time_s = 0.0
    train_loss_last = float('nan')
    val_loss_last = float('nan')

    best_val_loss = float('inf')
    best_state_cpu = None
    best_epoch = 0
    patience_left = EARLY_STOP_PATIENCE
    epochs_ran = 0
    early_stopped = False

    if USE_AMP:
        autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
    else:
        autocast_ctx = contextlib.nullcontext()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_train_loss = 0.0
        epoch_batches = 0
        t0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        micro = 0

        for batch in train_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                loss = loss_fn(logits, labels) / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            micro += 1
            epoch_train_loss += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
            epoch_batches += 1

            if micro % GRADIENT_ACCUMULATION_STEPS == 0:
                if MAX_GRAD_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        if micro % GRADIENT_ACCUMULATION_STEPS != 0:
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        train_time_s += time.perf_counter() - t0
        train_loss_last = epoch_train_loss / max(epoch_batches, 1)

        model.eval()
        probs_all = []
        val_loss_sum = 0.0
        val_batches = 0
        t1 = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
                labels = batch.pop('labels')
                with autocast_ctx:
                    logits = model(**batch).logits
                    probs = torch.sigmoid(logits)
                    val_loss_sum += float(loss_fn(logits, labels).item())
                probs_all.append(probs.float().cpu().numpy())
                val_batches += 1
        inference_time_s += time.perf_counter() - t1
        val_loss_last = val_loss_sum / max(val_batches, 1)
        epochs_ran = epoch

        print(f"  epoch {epoch}/{MAX_EPOCHS}  train_loss={train_loss_last:.4f}  val_loss={val_loss_last:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

        improved = val_loss_last < (best_val_loss - EARLY_STOP_MIN_DELTA)
        if improved:
            best_val_loss = val_loss_last
            best_epoch = epoch
            best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOP_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                early_stopped = True
                print(f"  early stopping at epoch {epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})")
                break

    if best_state_cpu is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_cpu.items()})

    model.eval()
    probs_all = []
    val_loss_sum = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                probs = torch.sigmoid(logits)
                val_loss_sum += float(loss_fn(logits, labels).item())
            probs_all.append(probs.float().cpu().numpy())
            val_batches += 1
    val_loss = val_loss_sum / max(val_batches, 1)

    prob_val = np.concatenate(probs_all, axis=0)
    pred_05 = (prob_val >= 0.5).astype(np.int64)
    per_label_05_df, summary_05 = multilabel_evaluation_report(y_val, pred_05, prob_val, LABEL_COLUMNS)

    best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
    pred_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
    per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val, pred_tuned, prob_val, LABEL_COLUMNS)

    y_true_flat = y_val.ravel()
    y_pred_05_flat = pred_05.ravel()
    y_pred_tuned_flat = pred_tuned.ravel()

    precision_micro_baseline = float(precision_score(y_true_flat, y_pred_05_flat, zero_division=0))
    recall_micro_baseline = float(recall_score(y_true_flat, y_pred_05_flat, zero_division=0))
    accuracy_micro_baseline = float(accuracy_score(y_true_flat, y_pred_05_flat))

    precision_micro_tuned = float(precision_score(y_true_flat, y_pred_tuned_flat, zero_division=0))
    recall_micro_tuned = float(recall_score(y_true_flat, y_pred_tuned_flat, zero_division=0))
    accuracy_micro_tuned = float(accuracy_score(y_true_flat, y_pred_tuned_flat))

    precision_macro_baseline = float(per_label_05_df['precision'].mean())
    recall_macro_baseline = float(per_label_05_df['recall'].mean())
    accuracy_macro_baseline = float(per_label_05_df['accuracy'].mean()) if 'accuracy' in per_label_05_df else float('nan')
    roc_auc_macro_baseline = float(np.nanmean(per_label_05_df['roc_auc'].to_numpy(dtype=np.float64)))

    precision_macro_tuned = float(per_label_tuned_df['precision'].mean())
    recall_macro_tuned = float(per_label_tuned_df['recall'].mean())
    accuracy_macro_tuned = float(per_label_tuned_df['accuracy'].mean()) if 'accuracy' in per_label_tuned_df else float('nan')
    roc_auc_macro_tuned = float(np.nanmean(per_label_tuned_df['roc_auc'].to_numpy(dtype=np.float64)))

    per_label_df = per_label_05_df.rename(
        columns={
            'precision': 'precision_baseline',
            'recall': 'recall_baseline',
            'f1': 'f1_baseline',
            'roc_auc': 'roc_auc_baseline',
            'accuracy': 'accuracy_baseline',
        }
    ).merge(
        per_label_tuned_df.rename(
            columns={
                'precision': 'precision_tuned',
                'recall': 'recall_tuned',
                'f1': 'f1_tuned',
                'roc_auc': 'roc_auc_tuned',
                'accuracy': 'accuracy_tuned',
            }
        ),
        on='label',
        how='left',
    )
    per_label_df['run_id'] = run_id
    per_label_df['preprocessing_variant'] = spec['preprocessing_variant']
    per_label_df['loss_variant'] = spec['loss_variant']

    thresholds_df['run_id'] = run_id
    thresholds_df['preprocessing_variant'] = spec['preprocessing_variant']
    thresholds_df['loss_variant'] = spec['loss_variant']

    conf_per_label_base_df, conf_agg_base_df, _ = make_confusion_artifacts(y_val, pred_05, LABEL_COLUMNS)
    conf_per_label_tuned_df, conf_agg_tuned_df, _ = make_confusion_artifacts(y_val, pred_tuned, LABEL_COLUMNS)

    conf_per_label_base_df['run_id'] = run_id
    conf_per_label_tuned_df['run_id'] = run_id
    conf_agg_base_df['run_id'] = run_id
    conf_agg_tuned_df['run_id'] = run_id

    row = {
        'run_id': run_id,
        'preprocessing_variant': spec['preprocessing_variant'],
        'loss_variant': spec['loss_variant'],
        'use_pos_weight': spec['use_pos_weight'],
        'pos_weight_clip_max': spec['pos_weight_clip_max'],
        'pos_weight_mean': pos_weight_mean,
        'pos_weight_max': pos_weight_max,
        'model_name': MODEL_NAME,
        'problem_type': PROBLEM_TYPE,
        'validation_fraction': VALIDATION_FRACTION,
        'random_state': RANDOM_STATE,
        'use_iterative_stratify': True,
        'rebalance_train': spec['rebalance_train'],
        'clean_to_toxic_ratio': spec['clean_to_toxic_ratio'],
        'rare_labels': 'severe_toxic|threat|identity_hate',
        'rare_oversample_factor': spec['rare_oversample_factor'],
        'max_copies_per_row': spec['max_copies_per_row'],
        'max_epochs': MAX_EPOCHS,
        'epochs_ran': epochs_ran,
        'best_epoch': best_epoch,
        'early_stopped': early_stopped,
        'warmup_ratio': WARMUP_RATIO,
        'warmup_steps': warmup_steps,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'early_stop_min_delta': EARLY_STOP_MIN_DELTA,
        'weight_decay': WEIGHT_DECAY,
        'grad_clip': MAX_GRAD_NORM,
        'grad_accum': GRADIENT_ACCUMULATION_STEPS,
        'use_bf16_amp': USE_AMP,
        'torch_compile': bool(USE_TORCH_COMPILE),
        'num_workers': NUM_WORKERS,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'num_train_rows': int(y_train.shape[0]),
        'num_val_rows': int(y_val.shape[0]),
        'num_parameters': num_params,
        'best_val_loss': best_val_loss,
        'train_loss': train_loss_last,
        'val_loss': val_loss,
        'train_time_s': train_time_s,
        'inference_time_s': inference_time_s,
        'f1_micro_baseline': summary_05['f1_micro'],
        'f1_macro_baseline': summary_05['f1_macro'],
        'f1_micro_tuned': summary_tuned['f1_micro'],
        'f1_macro_tuned': summary_tuned['f1_macro'],
        'precision_micro_baseline': precision_micro_baseline,
        'recall_micro_baseline': recall_micro_baseline,
        'accuracy_micro_baseline': accuracy_micro_baseline,
        'precision_macro_baseline': precision_macro_baseline,
        'recall_macro_baseline': recall_macro_baseline,
        'accuracy_macro_baseline': accuracy_macro_baseline,
        'roc_auc_macro_baseline': roc_auc_macro_baseline,
        'precision_micro_tuned': precision_micro_tuned,
        'recall_micro_tuned': recall_micro_tuned,
        'accuracy_micro_tuned': accuracy_micro_tuned,
        'precision_macro_tuned': precision_macro_tuned,
        'recall_macro_tuned': recall_macro_tuned,
        'accuracy_macro_tuned': accuracy_macro_tuned,
        'roc_auc_macro_tuned': roc_auc_macro_tuned,
    }

    pd.DataFrame([row]).to_csv(row_file, index=False)
    per_label_df.to_csv(per_label_file, index=False)
    thresholds_df.to_csv(thresholds_file, index=False)
    conf_per_label_base_df.to_csv(conf_per_label_base_file, index=False)
    conf_per_label_tuned_df.to_csv(conf_per_label_tuned_file, index=False)
    conf_agg_base_df.to_csv(conf_agg_base_file, index=False)
    conf_agg_tuned_df.to_csv(conf_agg_tuned_file, index=False)

    summary_rows.append(row)
    per_label_frames.append(per_label_df)
    threshold_frames.append(thresholds_df)
    conf_per_label_base_frames.append(conf_per_label_base_df)
    conf_per_label_tuned_frames.append(conf_per_label_tuned_df)
    conf_agg_base_rows.append(conf_agg_base_df.iloc[0].to_dict())
    conf_agg_tuned_rows.append(conf_agg_tuned_df.iloc[0].to_dict())

benchmark_summary_df = pd.DataFrame(summary_rows).sort_values(['preprocessing_variant', 'loss_variant']).reset_index(drop=True)
benchmark_per_label_df = pd.concat(per_label_frames, ignore_index=True)
benchmark_thresholds_df = pd.concat(threshold_frames, ignore_index=True)
benchmark_conf_per_label_baseline_df = pd.concat(conf_per_label_base_frames, ignore_index=True)
benchmark_conf_per_label_tuned_df = pd.concat(conf_per_label_tuned_frames, ignore_index=True)
benchmark_conf_agg_baseline_df = pd.DataFrame(conf_agg_base_rows).sort_values('run_id').reset_index(drop=True)
benchmark_conf_agg_tuned_df = pd.DataFrame(conf_agg_tuned_rows).sort_values('run_id').reset_index(drop=True)

benchmark_summary_df['f1_micro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_micro_tuned'] - benchmark_summary_df['f1_micro_baseline']
)
benchmark_summary_df['f1_macro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_macro_tuned'] - benchmark_summary_df['f1_macro_baseline']
)

summary_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_summary.csv'
per_label_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_per_label.csv'
thresholds_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_thresholds.csv'
conf_per_label_base_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_confusion_per_label_baseline.csv'
conf_per_label_tuned_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_confusion_per_label_tuned.csv'
conf_agg_base_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_confusion_aggregate_baseline.csv'
conf_agg_tuned_csv = ARTIFACT_DIR / 'distilbert_exp_d_benchmark_confusion_aggregate_tuned.csv'

benchmark_summary_df.to_csv(summary_csv, index=False)
benchmark_per_label_df.to_csv(per_label_csv, index=False)
benchmark_thresholds_df.to_csv(thresholds_csv, index=False)
benchmark_conf_per_label_baseline_df.to_csv(conf_per_label_base_csv, index=False)
benchmark_conf_per_label_tuned_df.to_csv(conf_per_label_tuned_csv, index=False)
benchmark_conf_agg_baseline_df.to_csv(conf_agg_base_csv, index=False)
benchmark_conf_agg_tuned_df.to_csv(conf_agg_tuned_csv, index=False)

print('Saved:')
for p in [
    summary_csv, per_label_csv, thresholds_csv,
    conf_per_label_base_csv, conf_per_label_tuned_csv,
    conf_agg_base_csv, conf_agg_tuned_csv,
]:
    print(' ', p)

display(benchmark_summary_df[[
    'run_id', 'preprocessing_variant', 'loss_variant',
    'f1_micro_baseline', 'f1_micro_tuned',
    'f1_macro_baseline', 'f1_macro_tuned',
    'roc_auc_macro_baseline', 'roc_auc_macro_tuned',
    'precision_micro_tuned', 'recall_micro_tuned', 'accuracy_micro_tuned',
    'train_time_s', 'inference_time_s', 'num_parameters',
    'f1_micro_delta_tuned_minus_baseline', 'f1_macro_delta_tuned_minus_baseline'
]])

In [ ]:
# Best run summary and selected hyperparameter traceability columns
best_idx = benchmark_summary_df['f1_macro_tuned'].idxmax()
best_row = benchmark_summary_df.loc[best_idx]

print('Best run by f1_macro_tuned')
print(best_row[['run_id', 'preprocessing_variant', 'loss_variant', 'f1_macro_tuned', 'f1_micro_tuned', 'best_val_loss']])

trace_cols = [
    'run_id', 'preprocessing_variant', 'loss_variant',
    'use_pos_weight', 'pos_weight_clip_max', 'pos_weight_mean', 'pos_weight_max',
    'max_length', 'batch_size', 'max_epochs', 'epochs_ran', 'best_epoch',
    'lr', 'weight_decay', 'warmup_ratio', 'warmup_steps',
    'grad_clip', 'grad_accum', 'early_stop_patience', 'early_stop_min_delta',
    'use_bf16_amp', 'torch_compile', 'num_workers',
    'num_train_rows', 'num_val_rows',
    'train_time_s', 'inference_time_s', 'num_parameters',
    'best_val_loss', 'f1_micro_tuned', 'f1_macro_tuned', 'roc_auc_macro_tuned'
]
display(benchmark_summary_df[trace_cols].sort_values('f1_macro_tuned', ascending=False))

In [ ]:
# Optional quick visuals
plt.figure(figsize=(10, 5))
plot_df = benchmark_summary_df.copy()
plot_df['x'] = plot_df['preprocessing_variant'] + ' | ' + plot_df['loss_variant']
plt.bar(plot_df['x'], plot_df['f1_macro_tuned'])
plt.xticks(rotation=35, ha='right')
plt.ylabel('F1 macro (tuned)')
plt.title('Experiment D: Tuned Macro F1 by Run Variant')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(plot_df['x'], plot_df['train_time_s'])
plt.xticks(rotation=35, ha='right')
plt.ylabel('Train time (s)')
plt.title('Experiment D: Training Time by Run Variant')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()